In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("01_raw_crm_input_15000.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (15000, 14)

Columns:
['interaction_id', 'crm_note_id', 'interaction_date', 'rep_id', 'hcp_id', 'city', 'region', 'territory_id', 'hcp_specialization', 'therapeutic_area', 'drug_id', 'drug_name', 'brand_name', 'crm_note']


In [ ]:
print("Missing CRM notes:", df["crm_note"].isna().sum())
print("Duplicate CRM notes:", df["crm_note"].duplicated().sum())
print("Empty CRM notes:", (df["crm_note"].fillna("").str.strip() == "").sum())


Missing CRM notes: 0
Duplicate CRM notes: 0
Empty CRM notes: 0


In [ ]:
df[["crm_note"]].head(10)

,crm_note
0,Saw HCP re Arthrelis. 2 patients mentioned inj...
1,Met on Arthrelis. HCP reports fewer tolerance ...
2,Met on Arthrelis. the previous adherence conce...
3,The HCP reviewed recent experience with Arthre...
4,Brief discussion focused on Rheumora. payer ap...
5,Quick f/u on Arthrelis. office reports fewer p...
6,Arthrelis discussion. use in routine practice ...
7,Saw HCP re Arthrelis. HCP remains cautious abo...
8,Rheumora discussion. recent approvals have gon...
9,Saw HCP re Renovia. practice wants a simpler d...


In [ ]:
# Display complete first 10 CRM notes

for i, note in enumerate(df["crm_note"].head(10), start=1):
    print(f"\n{'='*80}")
    print(f"CRM NOTE {i}")
    print(f"{'='*80}")
    print(note)


CRM NOTE 1
Saw HCP re Arthrelis. 2 patients mentioned injection-site reaction. recent outcomes have been favorable. Nothing else urgent.

CRM NOTE 2
Met on Arthrelis. HCP reports fewer tolerance complaints recently. several patients are not following the regimen consistently. Send patient-support material by Mar 06; owner REP0002.

CRM NOTE 3
Met on Arthrelis. the previous adherence concern has eased. HCP wants more confidence in expected benefit. authorization turnaround remains slow. HCP wants access / pa resource; REP0002 to f/u.

CRM NOTE 4
The HCP reviewed recent experience with Arthrelis during the scheduled visit. HCP is more comfortable with efficacy after seeing recent outcomes. Copay burden is hurting continuation. The HCP asked for affordability resource; follow-up ownership remains with REP0002.

CRM NOTE 5
Brief discussion focused on Rheumora. payer approval process is still too cumbersome.

CRM NOTE 6
Quick f/u on Arthrelis. office reports fewer payer problems this month

In [ ]:
# Create temporary word count for EDA

df["word_count_raw"] = df["crm_note"].str.split().str.len()

print("Average words :", round(df["word_count_raw"].mean(), 2))
print("Median words  :", df["word_count_raw"].median())
print("Minimum words :", df["word_count_raw"].min())
print("Maximum words :", df["word_count_raw"].max())

Average words : 25.69
Median words  : 24.0
Minimum words : 2
Maximum words : 73


In [ ]:
df["word_count_raw"].describe()

,word_count_raw
count,15000.000000
mean,25.689400
std,11.736086
min,2.000000
25%,17.000000
50%,24.000000
75%,33.000000
max,73.000000


In [ ]:
#Create a working copy
df_clean = df.copy()

In [ ]:
#Create text_raw
df_clean["text_raw"] = df_clean["crm_note"]

In [ ]:
#Lowercase
df_clean["text_lower"] = df_clean["text_raw"].str.lower()

In [ ]:
df_clean[["text_raw", "text_lower"]].head(10)

,text_raw,text_lower
0,Saw HCP re Arthrelis. 2 patients mentioned inj...,saw hcp re arthrelis. 2 patients mentioned inj...
1,Met on Arthrelis. HCP reports fewer tolerance ...,met on arthrelis. hcp reports fewer tolerance ...
2,Met on Arthrelis. the previous adherence conce...,met on arthrelis. the previous adherence conce...
3,The HCP reviewed recent experience with Arthre...,the hcp reviewed recent experience with arthre...
4,Brief discussion focused on Rheumora. payer ap...,brief discussion focused on rheumora. payer ap...
5,Quick f/u on Arthrelis. office reports fewer p...,quick f/u on arthrelis. office reports fewer p...
6,Arthrelis discussion. use in routine practice ...,arthrelis discussion. use in routine practice ...
7,Saw HCP re Arthrelis. HCP remains cautious abo...,saw hcp re arthrelis. hcp remains cautious abo...
8,Rheumora discussion. recent approvals have gon...,rheumora discussion. recent approvals have gon...
9,Saw HCP re Renovia. practice wants a simpler d...,saw hcp re renovia. practice wants a simpler d...


In [ ]:
#Normalize important abbreviations
import re

def normalize_abbreviations(text):
    # f/u = follow-up
    text = re.sub(r'\bf\s*/\s*u\b', 'follow up', text)

    # PA = prior authorization
    text = re.sub(r'\bpa\b', 'prior authorization', text)

    # "re" used in CRM notes = regarding
    text = re.sub(r'\bre\b', 'regarding', text)

    return text

df_clean["text_normalized"] = (
    df_clean["text_lower"]
    .apply(normalize_abbreviations)
)

In [ ]:
df_clean[
    ["text_lower", "text_normalized"]
].head(10)

,text_lower,text_normalized
0,saw hcp re arthrelis. 2 patients mentioned inj...,saw hcp regarding arthrelis. 2 patients mentio...
1,met on arthrelis. hcp reports fewer tolerance ...,met on arthrelis. hcp reports fewer tolerance ...
2,met on arthrelis. the previous adherence conce...,met on arthrelis. the previous adherence conce...
3,the hcp reviewed recent experience with arthre...,the hcp reviewed recent experience with arthre...
4,brief discussion focused on rheumora. payer ap...,brief discussion focused on rheumora. payer ap...
5,quick f/u on arthrelis. office reports fewer p...,quick follow up on arthrelis. office reports f...
6,arthrelis discussion. use in routine practice ...,arthrelis discussion. use in routine practice ...
7,saw hcp re arthrelis. hcp remains cautious abo...,saw hcp regarding arthrelis. hcp remains cauti...
8,rheumora discussion. recent approvals have gon...,rheumora discussion. recent approvals have gon...
9,saw hcp re renovia. practice wants a simpler d...,saw hcp regarding renovia. practice wants a si...


In [ ]:
#Remove administrative IDs and dates
def remove_admin_noise(text):
    # Remove representative IDs such as REP0002
    text = re.sub(r'\brep\d+\b', ' ', text, flags=re.IGNORECASE)

    # Remove dates such as "Mar 06", "Jun 11", etc.
    text = re.sub(
        r'\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s+\d{1,2}\b',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Remove standalone numbers
    text = re.sub(r'\b\d+\b', ' ', text)

    # Clean extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df_clean["text_no_admin"] = (
    df_clean["text_normalized"]
    .apply(remove_admin_noise)
)

In [ ]:
df_clean[
    ["text_normalized", "text_no_admin"]
].head(10)

,text_normalized,text_no_admin
0,saw hcp regarding arthrelis. 2 patients mentio...,saw hcp regarding arthrelis. patients mentione...
1,met on arthrelis. hcp reports fewer tolerance ...,met on arthrelis. hcp reports fewer tolerance ...
2,met on arthrelis. the previous adherence conce...,met on arthrelis. the previous adherence conce...
3,the hcp reviewed recent experience with arthre...,the hcp reviewed recent experience with arthre...
4,brief discussion focused on rheumora. payer ap...,brief discussion focused on rheumora. payer ap...
5,quick follow up on arthrelis. office reports f...,quick follow up on arthrelis. office reports f...
6,arthrelis discussion. use in routine practice ...,arthrelis discussion. use in routine practice ...
7,saw hcp regarding arthrelis. hcp remains cauti...,saw hcp regarding arthrelis. hcp remains cauti...
8,rheumora discussion. recent approvals have gon...,rheumora discussion. recent approvals have gon...
9,saw hcp regarding renovia. practice wants a si...,saw hcp regarding renovia. practice wants a si...


In [ ]:
#Remove punctuation
def remove_punctuation(text):
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean["text_no_punct"] = (
    df_clean["text_no_admin"].apply(remove_punctuation)
)

In [ ]:
df_clean[["text_no_admin", "text_no_punct"]].head(10)

,text_no_admin,text_no_punct
0,saw hcp regarding arthrelis. patients mentione...,saw hcp regarding arthrelis patients mentioned...
1,met on arthrelis. hcp reports fewer tolerance ...,met on arthrelis hcp reports fewer tolerance c...
2,met on arthrelis. the previous adherence conce...,met on arthrelis the previous adherence concer...
3,the hcp reviewed recent experience with arthre...,the hcp reviewed recent experience with arthre...
4,brief discussion focused on rheumora. payer ap...,brief discussion focused on rheumora payer app...
5,quick follow up on arthrelis. office reports f...,quick follow up on arthrelis office reports fe...
6,arthrelis discussion. use in routine practice ...,arthrelis discussion use in routine practice f...
7,saw hcp regarding arthrelis. hcp remains cauti...,saw hcp regarding arthrelis hcp remains cautio...
8,rheumora discussion. recent approvals have gon...,rheumora discussion recent approvals have gone...
9,saw hcp regarding renovia. practice wants a si...,saw hcp regarding renovia practice wants a sim...


In [ ]:
#Stopword handling
import nltk
nltk.download("stopwords")

from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
stop_words = set(stopwords.words("english"))

# Keep important negation words
words_to_keep = {
    "no", "not", "nor", "never",
    "neither", "without", "against"
}

stop_words = stop_words - words_to_keep

print("Number of stopwords:", len(stop_words))

Number of stopwords: 194


In [ ]:
important_terms = [
    "hcp", "patient", "patients",
    "safety", "efficacy", "adherence",
    "tolerance", "adverse",
    "dosage", "reimbursement",
    "insurance", "payer",
    "authorization", "competitor",
    "clinical", "evidence",
    "approval", "copay"
]

for word in important_terms:
    print(word, "→", word in stop_words)

hcp → False
patient → False
patients → False
safety → False
efficacy → False
adherence → False
tolerance → False
adverse → False
dosage → False
reimbursement → False
insurance → False
payer → False
authorization → False
competitor → False
clinical → False
evidence → False
approval → False
copay → False


In [ ]:
#Remove stopwords
def remove_stopwords(text):
    words = text.split()
    filtered_words = [
        word for word in words
        if word not in stop_words
    ]
    return " ".join(filtered_words)

df_clean["text_no_stopwords"] = (
    df_clean["text_no_punct"]
    .apply(remove_stopwords)
)

In [ ]:
df_clean[
    ["text_no_punct", "text_no_stopwords"]
].head(10)

,text_no_punct,text_no_stopwords
0,saw hcp regarding arthrelis patients mentioned...,saw hcp regarding arthrelis patients mentioned...
1,met on arthrelis hcp reports fewer tolerance c...,met arthrelis hcp reports fewer tolerance comp...
2,met on arthrelis the previous adherence concer...,met arthrelis previous adherence concern eased...
3,the hcp reviewed recent experience with arthre...,hcp reviewed recent experience arthrelis sched...
4,brief discussion focused on rheumora payer app...,brief discussion focused rheumora payer approv...
5,quick follow up on arthrelis office reports fe...,quick follow arthrelis office reports fewer pa...
6,arthrelis discussion use in routine practice f...,arthrelis discussion use routine practice feel...
7,saw hcp regarding arthrelis hcp remains cautio...,saw hcp regarding arthrelis hcp remains cautio...
8,rheumora discussion recent approvals have gone...,rheumora discussion recent approvals gone with...
9,saw hcp regarding renovia practice wants a sim...,saw hcp regarding renovia practice wants simpl...


In [ ]:
#Lemmatization
!pip -q install spacy
!python -m spacy download en_core_web_sm
import spacy

# Create a set of important pharmaceutical terms
protected_terms = set()

for column in ["drug_name", "brand_name", "competitor_brand"]:
    if column in df_clean.columns:
        terms = (
            df_clean[column]
            .dropna()
            .astype(str)
            .str.lower()
            .str.strip()
        )
        protected_terms.update(terms)

print("Number of protected terms:", len(protected_terms))
def lemmatize_text(text):
    doc = nlp(text)

    lemmas = []

    for token in doc:
        if not token.is_alpha:
            continue

        word = token.text.lower()

        # Keep drug/brand names unchanged
        if word in protected_terms:
            lemmas.append(word)
        else:
            lemmas.append(token.lemma_.lower())

    return " ".join(lemmas)


df_clean["clean_text"] = (
    df_clean["text_no_stopwords"]
    .apply(lemmatize_text)
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 123.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Number of protected terms: 57


In [ ]:
df_clean[
    ["text_no_stopwords", "clean_text"]
].head(10).to_string(index=False)

'                                                                                                                                                                                   text_no_stopwords                                                                                                                                                                              clean_text\n                                                                                saw hcp regarding arthrelis patients mentioned injection site reaction recent outcomes favorable nothing else urgent                                                                           see hcp regard arthrelis patient mention injection site reaction recent outcome favorable nothing else urgent\n                                               met arthrelis hcp reports fewer tolerance complaints recently several patients not following regimen consistently send patient support material owner                                       

In [ ]:
#Check 5 notes
for i in range(5):
    print(f"\n--- NOTE {i+1} ---")
    print("Original :", df_clean["crm_note"].iloc[i])
    print("Cleaned  :", df_clean["clean_text"].iloc[i])


--- NOTE 1 ---
Original : Saw HCP re Arthrelis. 2 patients mentioned injection-site reaction. recent outcomes have been favorable. Nothing else urgent.
Cleaned  : see hcp regard arthrelis patient mention injection site reaction recent outcome favorable nothing else urgent

--- NOTE 2 ---
Original : Met on Arthrelis. HCP reports fewer tolerance complaints recently. several patients are not following the regimen consistently. Send patient-support material by Mar 06; owner REP0002.
Cleaned  : meet arthrelis hcp report few tolerance complaint recently several patient not follow regiman consistently send patient support material owner

--- NOTE 3 ---
Original : Met on Arthrelis. the previous adherence concern has eased. HCP wants more confidence in expected benefit. authorization turnaround remains slow. HCP wants access / pa resource; REP0002 to f/u.
Cleaned  : meet arthrelis previous adherence concern ease hcp want confidence expect benefit authorization turnaround remain slow hcp want a

In [ ]:
#Check for empty cleaned notes
print(
    "Empty clean texts:",
    df_clean["clean_text"].str.strip().eq("").sum()
)

Empty clean texts: 0


In [ ]:
#Compare word counts
df_clean["clean_word_count"] = (
    df_clean["clean_text"].str.split().str.len()
)

print("Original average:",
      round(df_clean["word_count_raw"].mean(), 2))

print("Cleaned average:",
      round(df_clean["clean_word_count"].mean(), 2))

Original average: 25.69
Cleaned average: 17.3


In [ ]:
output_file = "CTS_CRM_15000_NLP_Preprocessed.csv"

df_clean.to_csv(output_file, index=False)

print("Saved successfully:", output_file)

Saved successfully: CTS_CRM_15000_NLP_Preprocessed.csv


In [ ]:
import os

print("File exists:", os.path.exists(output_file))
print("Rows:", len(df_clean))

File exists: True
Rows: 15000
